# Gemma Tutor — Personalized AI Coaching for Rajasthan Exam Aspirants

A self-hosted AI tutor that turns publisher-clean textbooks into source-preserved skill folders, letting any RAS Pre aspirant chat with a topic and generate verifiable mock tests grounded in the actual material.

**Built for:** Gemma 4 Good Hackathon (Kaggle, 2026-05-18)  
**Repo:** https://github.com/Mohit-5899/easilyclear  
**Demo video:** [3-min YouTube]  
**License:** Apache-2.0

## The problem

Pooja, 23, is preparing for the Rajasthan Administrative Services Preliminary exam in Sikar. RAS Pre is one of India's most competitive state exams — 1.6 million aspirants, ~700 seats. Coaching costs ₹35,000+ per year that her family can't afford.

Free PDFs exist but are watermarked, fragmented across coaching brands, and partly in Hindi. Off-the-shelf chatbots either don't know the syllabus or hallucinate without citations.

**Gemma Tutor** ingests publisher-clean textbooks (NCERT > RBSE) into a *subject-canonical* skill tree with verbatim source content — multiple sources merge into a single tree, brand names stay in internal frontmatter only — then lets the student chat with each topic and generate verifiable mock tests grounded in the actual textbook.

In [ ]:
# Setup — Kaggle has Tesseract pre-installed; we just pip-install Python deps.
!pip install -q PyMuPDF pytesseract rank-bm25 pydantic python-frontmatter sentence-transformers httpx

import os
from pathlib import Path

# Set your OpenRouter API key as a Kaggle Secret named OPENROUTER_API_KEY
from kaggle_secrets import UserSecretsClient  # type: ignore
try:
    os.environ["OPENROUTER_API_KEY"] = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
except Exception:
    os.environ.setdefault("OPENROUTER_API_KEY", "<paste-your-key-here>")
print("Setup complete.")

## Step 1: Inspect a pre-ingested subject tree

We've pre-ingested the Rajasthan Geography subject tree (267-page source, 1061 paragraphs after OCR, 13 chapters, 100% paragraph coverage). Each leaf is a Markdown file with YAML frontmatter and verbatim source content.

The tree is **subject-canonical**: there's one tree per subject, not per book. When NCERT Class 11 lands as a second source, its leaves merge into this same tree (via cosine + Gemma judge), with each source kept distinct in the leaf's `sources[]` list and a separate `## Source N` body section.

In [ ]:
!git clone -q https://github.com/Mohit-5899/easilyclear /kaggle/working/gemma-tutor
import os
os.chdir("/kaggle/working/gemma-tutor")

# Show the chapter structure
!find database/skills/rajasthan_geography -maxdepth 1 -type d | sort | head -20

In [ ]:
# Read one leaf to see the frontmatter (multi-source) + body (verbatim).
import frontmatter
from pathlib import Path

leaf = Path("database/skills/rajasthan_geography/02-physiographic-divisions/03-characteristics-and-divisions-of-aravali.md")
post = frontmatter.load(leaf)

print("=== Frontmatter (sources[] is what makes the tree multi-publisher) ===")
for k, v in post.metadata.items():
    print(f"  {k}: {v}")

print("\n=== Body (first 800 chars; bodies are verbatim source paragraphs) ===")
print(post.content[:800])

## Step 2: Tutor Q&A — answer with source citations

The production tutor is an **agent** that calls a `lookup_skill_content` tool against the canonical subject tree, then streams the answer with `[N]` citation markers tied to the retrieved paragraphs. For a one-cell illustration we'll call the BM25 retriever directly — same paragraphs the agent would see, minus the tool-calling envelope.

In [ ]:
import sys, asyncio
sys.path.insert(0, "backend")

from tutor.retriever import build_retriever_for_node
from tutor.prompt import build_tutor_messages
from llm.openrouter import OpenRouterClient
from llm.base import Message

node_id = (
    "rajasthan_geography/02-physiographic-divisions/"
    "03-characteristics-and-divisions-of-aravali"
)
retriever = build_retriever_for_node(Path("database/skills"), node_id)
hits = retriever.search("Why is Aravalli called the planning region?", k=3)
print(f"Retrieved {len(hits)} paragraphs:")
for h in hits:
    print(f"  [score={h.score:.2f} page={h.page}] {h.snippet[:120]}…")

messages = build_tutor_messages(
    question="Why is Aravalli called the planning region?",
    node_title="Aravalli Mountain Range",
    hits=hits,
)

client = OpenRouterClient(api_key=os.environ["OPENROUTER_API_KEY"])
response = asyncio.run(client.complete(
    [Message(**m) for m in messages],
    model="google/gemma-4-26b-a4b-it",
    temperature=0.3,
    max_tokens=400,
))
print("\n=== Tutor answer ===")
print(response.content)

## Step 3: Generate a verifiable mock test

The mock test generator runs three stages:

1. **Generation** — Gemma 4 26B emits structured MCQs in JSON mode, with an `answer_span` field that's a verbatim substring of the cited paragraph
2. **Span verification** (deterministic, no LLM) — confirms `answer_span` actually appears in the source
3. **LLM judge** — confirms each question has a single correct answer and isn't solvable from general knowledge alone

We oversample to 8 candidates and keep the 5 that pass.

In [ ]:
from tests_engine.orchestrator import build_mock_test

# Reuse the retriever's paragraph list as the corpus.
paragraphs = list(getattr(retriever, "_paragraphs", []))

test = asyncio.run(build_mock_test(
    llm=client,
    generator_model="google/gemma-4-26b-a4b-it",
    judge_model="google/gemma-4-26b-a4b-it",
    node_id=node_id,
    paragraphs=paragraphs,
    n=5,
    oversample_n=8,
    difficulty_mix=(2, 2, 1),
))

print(f"Generated {len(test.questions)} questions in {test.elapsed_seconds:.1f}s\n")
for i, q in enumerate(test.questions, 1):
    print(f"Q{i} [{q.difficulty}]: {q.prompt}")
    for k, v in q.choices.items():
        marker = "✓" if k == q.correct else " "
        print(f"  {marker} {k}: {v}")
    print(f"  Source span: \"{q.answer_span[:80]}…\"")
    print()

## How this scales

The same pipeline works for any English-language textbook with a clean source:

- **NCERT Class 11 / 12** — pre-structured PDFs, ~10 min ingest on the paid Gemma 4 26B tier; auto-merges into the existing subject tree via Stage 6.5b
- **RBSE Class 10 / 12** — analogous structure for Rajasthan State Board
- **Patwari + REET** — different exam-coverage tags, same code

Per-student cost is ₹0 once self-hosted (Ollama + commodity hardware). Compare to ₹35K/year coaching.

To add a new source: drop the PDF into `/admin`, fill the metadata form, watch the 8 pipeline stages stream live. The merged subject tree appears in `/library/<subject>` when done.

## Try the live demo

- **Hosted demo**: [GCP Cloud Run url — pending] (read-only, pre-ingested with the Rajasthan Geography subject tree)
- **GitHub**: https://github.com/Mohit-5899/easilyclear
- **Run locally**: see the repo `README.md` Quickstart (Python 3.12, Node 22, `OPENROUTER_API_KEY`).

Built solo over 33 days. Every implementation choice has a research note or spec — check `docs/research/` and `docs/superpowers/specs/`.

The hardest single decision was **source preservation over summarization**; everything else followed from that.